<a href="https://colab.research.google.com/github/jwliu24/Uber-Operational-Analysis/blob/main/Week_4_Hypothesis_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, ttest_ind

In [2]:
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Colab Notebooks/uber_rides_data_processed.csv'
df = pd.read_csv(file_path)
df.head()

Mounted at /content/drive


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Payment Method,is_successful,is_cancelled_customer,is_cancelled_driver,is_incomplete,DateTime,Hour,DayOfWeek,Month,IsWeekend
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,False,False,False,False,2024-03-23 12:29:38,12,Saturday,3,True
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,UPI,False,False,False,True,2024-11-29 18:01:39,18,Friday,11,False
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,Debit Card,True,False,False,False,2024-08-23 08:56:10,8,Friday,8,False
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,UPI,True,False,False,False,2024-10-21 17:17:25,17,Monday,10,False
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,UPI,True,False,False,False,2024-09-16 22:08:00,22,Monday,9,False


SECTION 1: Chi-Square Test
Weekend vs Cancellation
===============================
Step 1: Define Cancellation Flag

In [3]:
df['is_cancelled'] = (
    df['is_cancelled_customer'] | df['is_cancelled_driver']
)

Step 2: Contingency Table

In [4]:
contingency_table = pd.crosstab(df['IsWeekend'], df['is_cancelled'])
contingency_table

is_cancelled,False,True
IsWeekend,,
False,80231,26829
True,32269,10671


Step 3: Run Chi-Square Test

In [5]:
chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)

Chi-square statistic: 0.7016946388858991
Degrees of freedom: 1
p-value: 0.4022148562568606


Step 4: Interpretation

In [6]:
if p < 0.05:
    print("Statistically significant relationship detected.")
else:
    print("No statistically significant relationship detected.")

No statistically significant relationship detected.



SECTION 2: Two-Sample T-Test
Booking Value Comparison

Step 1: Prepare Data

In [20]:
completed = df[df['Booking Status'] == 'Completed']['Avg VTAT'].dropna()
cancelled = df[df['Booking Status'] == 'Cancelled by Customer']['Avg VTAT'].dropna()

print("Size of completed group:", len(completed))
print("Size of cancelled group:", len(cancelled))

Size of completed group: 93000
Size of cancelled group: 10500


Step 2: Compare Means

In [21]:
print("\nCompleted mean Wait Time (mins):", round(completed.mean(), 2))
print("Cancelled mean Wait Time (mins):", round(cancelled.mean(), 2))


Completed mean Wait Time (mins): 8.51
Cancelled mean Wait Time (mins): 12.51


Step 3: Run T-Test

In [22]:
t_stat, p_value = ttest_ind(completed, cancelled, equal_var=False)

print("T-statistic:", t_stat)
print("p-value:", p_value)

T-statistic: -90.62644369519307
p-value: 0.0


Step 4: Interpretation

In [23]:
if p_value < 0.05:
    print("Statistically significant difference in booking value.")
else:
    print("No statistically significant difference in booking value.")

Statistically significant difference in booking value.


OPTIONAL: Vehicle Type vs Cancellation

In [24]:
vehicle_table = pd.crosstab(df['Vehicle Type'], df['is_cancelled'])

chi2, p, dof, expected = chi2_contingency(vehicle_table)

print("p-value:", p)

p-value: 0.8908299999794936


### Chi-Square Test Conclusion

**Hypothesis:** Is there a statistically significant relationship between the day of the week (Weekday vs. Weekend) and the overall cancellation rate?

**Results:**
* We ran a Chi-Square Test of Independence comparing the total cancellation rates (both customer and driver cancellations) on Weekdays versus Weekends.
* The test returned a **p-value of 0.402**, which is well above our 0.05 significance threshold.

**Conclusion:** We fail to reject the null hypothesis. There is no statistically significant relationship between whether a ride occurs on a weekend or a weekday and whether it gets cancelled. The overall proportion of rides that get cancelled is practically identical regardless of the day of the week (~25%).

*(Note: While the overall daily volume remains constant, our hourly heatmap analysis reveals that the specific time of day cancellations occur fluctuates heavily on weekdays compared to weekends).*

### T-Test Conclusion

**Hypothesis:** Is there a statistically significant difference in the average wait time (VTAT) between rides that are completed and rides that are cancelled by the customer?

**Results:**
* We ran an independent Two-Sample T-Test comparing the `Avg VTAT` of Completed rides vs. Cancelled rides.
* The test returned a **p-value of 0.0**, which is well below our 0.05 significance threshold.

**Conclusion:** We reject the null hypothesis. There is a highly statistically significant difference in wait times. Customers who cancel their rides experience significantly longer wait times than customers whose rides are successfully completed.